# Programação para Dados — Fase I
**Catálogo de jogos (Steam)** · consultas com a biblioteca `jogos` (sem pandas/numpy)

Estrutura do projeto:
- `jogos/modelos.py` — `Jogo` e `FaixaPreco` (dados do domínio, independentes do distribuidor)
- `jogos/leitores.py` — `LeitorBase` / `LeitorSteamCSV` (única parte que conhece o layout do CSV)
- `jogos/catalogo.py` — `CatalogoJogos` (consultas)
- `jogos/excecoes.py` — exceções próprias
- `tests/test_catalogo.py` — testes automatizados sobre a amostra de 20 jogos

Para suportar Epic ou Paradox basta criar outro leitor; as consultas não mudam.

In [1]:
from pathlib import Path
from jogos import CatalogoJogos

RAIZ = Path.cwd()
ARQ_COMPLETO = RAIZ / "data" / "steam_games.csv"
ARQ_AMOSTRA = RAIZ / "data" / "amostra_20_jogos.csv"

## 1. Testes automatizados (amostra de 20 jogos × cálculo manual em planilha)

In [2]:
import unittest
suite = unittest.defaultTestLoader.discover(str(RAIZ / "tests"), top_level_dir=str(RAIZ))
resultado = unittest.TextTestRunner(verbosity=1).run(suite)
assert resultado.wasSuccessful()

.

.

.

.

.

.

... (11 outputs left out)

.

.

.

----------------------------------------------------------------------
Ran 20 tests in 0.030s

OK


## 2. Funções de apresentação (mesmo código para amostra e arquivo completo)

In [3]:
def responder(caminho):
    catalogo = CatalogoJogos.de_csv_steam(caminho)
    print(f"Arquivo: {Path(caminho).name} — {len(catalogo)} jogos\n")

    # Pergunta 1
    perc, cont = catalogo.percentual_gratuitos_pagos(), catalogo.contagem_gratuitos_pagos()
    print("P1) Percentual de jogos gratuitos e pagos")
    print(f"    Gratuitos: {perc['gratuitos']:.2f}% ({cont['gratuitos']} jogos)")
    print(f"    Pagos:     {perc['pagos']:.2f}% ({cont['pagos']} jogos)\n")

    # Pergunta 2
    por_ano, anos = catalogo.lancamentos_por_ano(), catalogo.anos_com_mais_lancamentos()
    print(f"P2) Ano(s) com mais lançamentos: {anos} — {por_ano[anos[0]]} jogos")
    print("    Lançamentos por ano:")
    maior = max(por_ano.values())
    for ano, qtd in por_ano.items():
        print(f"    {ano}  {qtd:>5}  {'#' * round(30 * qtd / maior)}")
    print()

    # Pergunta 3 (própria)
    print("P3) Aprovação dos jogadores (% de avaliações positivas) por faixa de preço")
    print(f"    {'Faixa':<16}{'Jogos':>7}{'Avaliados':>11}{'Positivas':>11}{'Negativas':>11}{'Aprovação':>11}")
    for r in catalogo.aprovacao_por_faixa_de_preco():
        aprov = "—" if r.aprovacao is None else f"{r.aprovacao:.2f}%"
        print(f"    {r.faixa:<16}{r.jogos:>7}{r.jogos_avaliados:>11}{r.positivas:>11}{r.negativas:>11}{aprov:>11}")

## 3. Respostas sobre o arquivo completo

In [4]:
responder(ARQ_COMPLETO)

Arquivo: steam_games.csv — 5199 jogos

P1) Percentual de jogos gratuitos e pagos
    Gratuitos: 16.37% (851 jogos)
    Pagos:     83.63% (4348 jogos)

P2) Ano(s) com mais lançamentos: [2021] — 1048 jogos
    Lançamentos por ano:
    2003      2  
    2005      1  
    2006      6  
    2007     10  
    2008     18  #
    2009     22  #
    2010     36  #
    2011     19  #
    2012     28  #
    2013     41  #
    2014    128  ####
    2015    218  ######
    2016    383  ###########
    2017    505  ##############
    2018    660  ###################
    2019    648  ###################
    2020    822  ########################
    2021   1048  ##############################
    2022    604  #################

P3) Aprovação dos jogadores (% de avaliações positivas) por faixa de preço
    Faixa             Jogos  Avaliados  Positivas  Negativas  Aprovação
    Gratuito            851        558    1007508     209363     82.79%
    Até US$ 10         3315       3021    2158703     19965

## 4. Respostas sobre a amostra (conferidas contra a planilha `docs/calculo_manual_amostra.xlsx`)

In [5]:
responder(ARQ_AMOSTRA)

Arquivo: amostra_20_jogos.csv — 20 jogos

P1) Percentual de jogos gratuitos e pagos
    Gratuitos: 10.00% (2 jogos)
    Pagos:     90.00% (18 jogos)

P2) Ano(s) com mais lançamentos: [2021] — 5 jogos
    Lançamentos por ano:
    2009      2  ############
    2013      1  ######
    2014      1  ######
    2015      1  ######
    2016      1  ######
    2017      2  ############
    2018      1  ######
    2019      1  ######
    2020      2  ############
    2021      5  ##############################
    2022      3  ##################

P3) Aprovação dos jogadores (% de avaliações positivas) por faixa de preço
    Faixa             Jogos  Avaliados  Positivas  Negativas  Aprovação
    Gratuito              2          1         26         15     63.41%
    Até US$ 10           10          9        742        229     76.42%
    US$ 10 a 30           8          8       5550       1004     84.68%
    Acima de US$ 30       0          0          0          0          —


## 5. Facilidade de modificação
Uma nova faixa de preço, por exemplo, não exige alterar a biblioteca:

In [6]:
from jogos import FaixaPreco
inf = float("inf")
catalogo = CatalogoJogos.de_csv_steam(ARQ_COMPLETO)
for r in catalogo.aprovacao_por_faixa_de_preco([FaixaPreco("Até US$ 5", 0, 5), FaixaPreco("Acima de US$ 5", 5, inf)]):
    print(r.faixa, r.jogos, f"{r.aprovacao:.2f}%")

Até US$ 5 2218 87.74%
Acima de US$ 5 2130 86.75%
